# TM-Time Tracking Notebook

This Jupyter Notebook is created to document the code and compliment additional details about the ETL pipeline covered in `etl.py` that might be missing in the code I uploaded to the Github repository.

Please note that the code in this notebook may slightly from that in `etl.py`.

In [1]:
import pandas as pd
import numpy as np
import time

In [2]:
pd.set_option('display.max_rows', 100)

In [3]:
from langdetect import detect

In [4]:
from deep_translator import GoogleTranslator

## Ingestion Layer

This layer is responsible for ingesting a `.csv` dataset and later breaks this dataset into chunks. From my experiment, chunking is required to ensure that the translator function works properly for all rows.

Based on my experiment, the best chunk size is 500.

To preserve the original version of the dataset, I create a variable `raw_df` as a dataframe for the raw data.

In [6]:
raw_df = pd.read_csv('dailycheckins.csv')

In [7]:
## Split a single large dataframe into small ones
chunk_size = 500
raw_df_list = [raw_df.iloc[i:i + chunk_size] for i in range(0, len(raw_df), chunk_size)]


In [8]:
raw_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20500 entries, 0 to 20499
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   user       20495 non-null  object 
 1   timestamp  20500 non-null  object 
 2   hours      20500 non-null  float64
 3   project    20500 non-null  object 
dtypes: float64(1), object(3)
memory usage: 640.8+ KB


Each chunk consists of a dataframe with the same columns but the number of rows is only 500.

In [10]:
raw_df_list[0].shape

(500, 4)

Originally, there are 20,500 rows of the data with 4 columns : user, timestamp, hours and project 

In [12]:
raw_df.shape

(20500, 4)

Find the null values of all users

In [14]:
raw_df.user.isna().sum()

5

There are 5 rows with `null` value in `user` column.

In [16]:
raw_df[raw_df.user.isna()]

,user,timestamp,hours,project
15797,NaN,2017-12-27 10:36:14.000121 UTC,4.00,project-40
15798,NaN,2017-12-27 10:36:14.000121 UTC,3.00,learning
17572,NaN,2017-10-12 10:31:44.000227 UTC,2.75,project-47
17573,NaN,2017-10-12 10:31:44.000227 UTC,4.00,bizdev
17574,NaN,2017-10-12 10:31:44.000227 UTC,1.00,transit


No null value in timestamp column and hours column

In [18]:
raw_df.timestamp.isna().sum()

0

In [19]:
raw_df.hours.isna().sum()

0

In [20]:
raw_df.project.isna().sum()  # no null values in project column

0

There are 144 unique projects

In [22]:
raw_df.project.nunique()

144

There are 40 active users in this dataset

In [24]:
raw_df.user.nunique()

40

## Transformation Layer

This layer is the processing layer responsible for cleaning the dataset that we explored earlier. So far, we have identified the following flaws:

- Null values in `user` column
- Columns need to be rearraged to make the report looks more readable. For instance, `hours` column should be located next to `timestamp` column
- Various datetime format in `timestamp` column
- Some rows in `timestamp` column contains non-English characters




First, create a function that translates a non-English text into an English text

In [28]:
def detect_and_translate(text):
    if pd.isna(text):
        return text

    try:
        # Detect language
        lang = detect(str(text))
        #print(lang)

        if lang in ['ru', 'uk', 'bg']:
            translator = GoogleTranslator(source='auto', target='en')
            return translator.translate(text)
        return text
            
    except Exception as e:
        raise Exception

Next, copy a list of raw dataframe ingested from the file to the new one called `transformed_df_list`.

#### Stage 1: Impute null values in `user` column with a default string value : `unknown` 


Copy the raw dataframe to the new one

In [32]:
transformed_df = raw_df.copy(deep=True)

Then, impute null values in `user` column

In [34]:
transformed_df.user.fillna('unknown user', inplace=True)

/var/folders/b3/rsjzc5l10ggc4wvt6py_14140000gn/T/ipykernel_77246/463104869.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  transformed_df.user.fillna('unknown user', inplace=True)


Confirm that there is no null value left

In [36]:
transformed_df.user.isna().sum()

0

#### Stage 2: Rearrange the columns

- hours column should be located next to timestamp column

In [38]:
new_ordered_col = ['user', 'project', 'hours', 'timestamp']

In [39]:
transformed_df = transformed_df[new_ordered_col]

In [40]:
transformed_df_list = [transformed_df.iloc[i:i + chunk_size] for i in range(0, len(transformed_df), chunk_size)]

In [41]:
transformed_df_list[1]

,user,project,hours,timestamp
500,sansa,project-31,2.0,2018-11-19 00:00:00 UTC
501,sansa,project-51,1.0,2018-11-19 00:00:00 UTC
502,sansa,project-32,1.0,2018-11-19 00:00:00 UTC
503,sansa,bizdev,3.0,2018-11-19 00:00:00 UTC
504,robert,opsandadmin,1.0,11/19/2018 12:00 AM
...,...,...,...,...
995,viserys,opsandadmin,0.5,11/09/2018 11:24 AM
996,robb,learning,6.0,2018-11-09 11:00:50.1545 UTC
997,robb,cultureandmanagement,2.0,2018-11-09 11:00:50.1545 UTC
998,jeor,learning,0.5,2018-11-09 10:59:51.1543 UTC


#### Stage 3: Translate non-English text in `timestamp` column into English text

In [43]:
import time

start = time.perf_counter()

#---------------
count = 0

for df in transformed_df_list:
    
    print(f"Data frame : {count}")
    
    df['trans_timestamp'] = df['timestamp'].apply(detect_and_translate)
    
    count= count+1
#--------------

end = time.perf_counter()
processing_time = end - start
print(f"Processing time: {processing_time} seconds")


Data frame : 0


/var/folders/b3/rsjzc5l10ggc4wvt6py_14140000gn/T/ipykernel_77246/2633248873.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['trans_timestamp'] = df['timestamp'].apply(detect_and_translate)


Data frame : 1


/var/folders/b3/rsjzc5l10ggc4wvt6py_14140000gn/T/ipykernel_77246/2633248873.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['trans_timestamp'] = df['timestamp'].apply(detect_and_translate)


Data frame : 2


...

/var/folders/b3/rsjzc5l10ggc4wvt6py_14140000gn/T/ipykernel_77246/2633248873.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['trans_timestamp'] = df['timestamp'].apply(detect_and_translate)


Data frame : 39


/var/folders/b3/rsjzc5l10ggc4wvt6py_14140000gn/T/ipykernel_77246/2633248873.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['trans_timestamp'] = df['timestamp'].apply(detect_and_translate)


Data frame : 40
Processing time: 314.3568823339883 seconds


/var/folders/b3/rsjzc5l10ggc4wvt6py_14140000gn/T/ipykernel_77246/2633248873.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['trans_timestamp'] = df['timestamp'].apply(detect_and_translate)


#### Stage 4: Convert values in `trans_timestamp` column into those of `datetime` type in the new column: `datetime_col` 

In [45]:
for df in transformed_df_list:
    
    print(f"Data frame : {count}")

    try:
           
        df['datetime_col'] = pd.to_datetime(df['trans_timestamp'], format="mixed")

    except Exception:
        
        raise(f"Failed to convert to datetime column at batch {count} due to {Exception}")
   
    count= count+1

Data frame : 41
Data frame : 42
Data frame : 43
Data frame : 44
Data frame : 45
Data frame : 46
Data frame : 47
Data frame : 48
Data frame : 49
Data frame : 50
Data frame : 51
Data frame : 52
Data frame : 53
Data frame : 54
Data frame : 55
Data frame : 56
Data frame : 57
Data frame : 58
Data frame : 59
Data frame : 60
Data frame : 61
Data frame : 62
Data frame : 63
Data frame : 64
Data frame : 65
Data frame : 66
Data frame : 67
Data frame : 68
Data frame : 69
Data frame : 70
Data frame : 71
Data frame : 72


/var/folders/b3/rsjzc5l10ggc4wvt6py_14140000gn/T/ipykernel_77246/1249685214.py:7: FutureWarning: In a future version of pandas, parsing datetimes with mixed time zones will raise an error unless `utc=True`. Please specify `utc=True` to opt in to the new behaviour and silence this warning. To create a `Series` with mixed offsets and `object` dtype, please use `apply` and `datetime.datetime.strptime`
  df['datetime_col'] = pd.to_datetime(df['trans_timestamp'], format="mixed")
/var/folders/b3/rsjzc5l10ggc4wvt6py_14140000gn/T/ipykernel_77246/1249685214.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['datetime_col'] = pd.to_datetime(df['trans_timestamp'], format="mixed")
/var/folders/b3/rsjzc5l10ggc4wvt6py_14140000gn/T/ipykernel_77246/124968521

Data frame : 73
Data frame : 74
Data frame : 75
Data frame : 76
Data frame : 77
Data frame : 78
Data frame : 79
Data frame : 80
Data frame : 81


/var/folders/b3/rsjzc5l10ggc4wvt6py_14140000gn/T/ipykernel_77246/1249685214.py:7: FutureWarning: In a future version of pandas, parsing datetimes with mixed time zones will raise an error unless `utc=True`. Please specify `utc=True` to opt in to the new behaviour and silence this warning. To create a `Series` with mixed offsets and `object` dtype, please use `apply` and `datetime.datetime.strptime`
  df['datetime_col'] = pd.to_datetime(df['trans_timestamp'], format="mixed")
/var/folders/b3/rsjzc5l10ggc4wvt6py_14140000gn/T/ipykernel_77246/1249685214.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['datetime_col'] = pd.to_datetime(df['trans_timestamp'], format="mixed")
/var/folders/b3/rsjzc5l10ggc4wvt6py_14140000gn/T/ipykernel_77246/124968521

#### Stage 5: Final Cleaning
- Drop origninal timestamp columns
- Rename the latest timestamp column as timestamp column

In [47]:
start = time.perf_counter()

count = 0

for df in transformed_df_list:

    print(f"dataframe : {count}")

    # Drop columns
    df.drop(columns=['timestamp', 'trans_timestamp'], inplace=True)

    count = count+1

end = time.perf_counter()
processing_time = end - start
print(f"Processing time: {processing_time} seconds")

dataframe : 0
dataframe : 1
dataframe : 2
dataframe : 3
dataframe : 4
dataframe : 5
dataframe : 6
dataframe : 7
dataframe : 8
dataframe : 9
dataframe : 10
dataframe : 11
dataframe : 12
dataframe : 13
dataframe : 14
dataframe : 15
dataframe : 16
dataframe : 17
dataframe : 18
dataframe : 19
dataframe : 20
dataframe : 21
dataframe : 22
dataframe : 23
dataframe : 24
dataframe : 25
dataframe : 26
dataframe : 27
dataframe : 28
dataframe : 29
dataframe : 30
dataframe : 31
dataframe : 32
dataframe : 33
dataframe : 34
dataframe : 35
dataframe : 36
dataframe : 37
dataframe : 38
dataframe : 39
dataframe : 40
Processing time: 0.0319232500041835 seconds


/var/folders/b3/rsjzc5l10ggc4wvt6py_14140000gn/T/ipykernel_77246/927256398.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.drop(columns=['timestamp', 'trans_timestamp'], inplace=True)
/var/folders/b3/rsjzc5l10ggc4wvt6py_14140000gn/T/ipykernel_77246/927256398.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.drop(columns=['timestamp', 'trans_timestamp'], inplace=True)
/var/folders/b3/rsjzc5l10ggc4wvt6py_14140000gn/T/ipykernel_77246/927256398.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pand

In [48]:
start = time.perf_counter()

count = 0

for df in transformed_df_list:

    print(f"dataframe : {count}")

    # Rename datetime_col as timestamp
    df.rename(columns={"datetime_col" : "timestamp"}, inplace=True, errors="raise")
  
    count = count+1

end = time.perf_counter()
processing_time = end - start
print(f"Processing time: {processing_time} seconds")

dataframe : 0
dataframe : 1
dataframe : 2
dataframe : 3
dataframe : 4
dataframe : 5
dataframe : 6
dataframe : 7
dataframe : 8
dataframe : 9
dataframe : 10
dataframe : 11
dataframe : 12
dataframe : 13
dataframe : 14
dataframe : 15
dataframe : 16
dataframe : 17
dataframe : 18
dataframe : 19
dataframe : 20
dataframe : 21
dataframe : 22
dataframe : 23
dataframe : 24
dataframe : 25
dataframe : 26
dataframe : 27
dataframe : 28
dataframe : 29
dataframe : 30
dataframe : 31
dataframe : 32
dataframe : 33
dataframe : 34
dataframe : 35
dataframe : 36
dataframe : 37
dataframe : 38
dataframe : 39
dataframe : 40
Processing time: 0.0018935000407509506 seconds


/var/folders/b3/rsjzc5l10ggc4wvt6py_14140000gn/T/ipykernel_77246/1497639031.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.rename(columns={"datetime_col" : "timestamp"}, inplace=True, errors="raise")


## Load Layer

Sqlalchemy Solution

In [57]:
from sqlalchemy import create_engine, text

sqlite_engine = create_engine('sqlite:///data_checkins.db')

In [59]:
for df in transformed_df_list:
    df.to_sql('data_check_ins', sqlite_engine, if_exists='append', index=False)

Query all distinct users from data_check table 

In [61]:
with sqlite_engine.connect() as conn:
    exe = conn.execute(text('SELECT DISTINCT(user) FROM data_check_ins ORDER BY user DESC'))
    result = exe.fetchall()
    result_tuple = tuple(name for (name,) in result)

print(result)

[('ygritte',), ('viserys',), ('varys',), ('unknown user',), ('tywin',), ('tyrion',), ('tormund',), ('tommen',), ('theon',), ('talisa',), ('stannis',), ('shae',), ('sansa',), ('samwell',), ('robert',), ('robb',), ('ramsay',), ('ned',), ('missandei',), ('melisandre',), ('margaery',), ('littlefinger',), ('khal',), ('jorah',), ('jon',), ('joffrey',), ('jeor',), ('jaime',), ('hound',), ('gilly',), ('gendry',), ('ellaria',), ('davos',), ('daenerys',), ('daario',), ('cersei',), ('catelyn',), ('bronn',), ('brienne',), ('bran',), ('arya',)]


Check if there are 41 users ( 40 valid users and 1 unknown user ) as we expected

In [64]:
with sqlite_engine.connect() as conn:
    exe = conn.execute(text('SELECT COUNT(DISTINCT(user)) FROM data_check_ins ORDER BY user DESC'))
    result = exe.fetchall()
    result_tuple = tuple(name for (name,) in result)

print(result)

[(41,)]
